# Обучение голоса в Kaggle — фоновый прогон

**Запускать так: Save Version → Save & Run All (Commit).** Ноутбук выполнится
целиком на серверах Kaggle: браузер, вкладку и ноутбук можно закрыть, обрыв
связи ни на что не влияет. Интерактивная сессия (Run All в редакторе), наоборот,
умирает вместе с соединением, и несохранённый результат теряется.

Перед запуском в панели справа: **Accelerator → GPU**, **Internet → On**
(интернет доступен только на аккаунте с подтверждённым телефоном), и
**+ Add Data** — датасет с вашей записью.

Лимиты: GPU-сессия до 9 часов, 30 часов GPU в неделю (сброс в субботу 00:00 UTC),
до 20 ГБ в output версии.

Ноутбук устроен так, чтобы его можно было запускать сколько угодно раз подряд:
он сам определяет, начинает с нуля или продолжает с чекпоинта прошлой версии,
и сам останавливает обучение до лимита сессии, чтобы результат успел сохраниться.


## Параметры


In [ ]:
SPEAKER = 'anna'          # имя голоса
VC = 'rvc'                # 'rvc' (Applio) или 'sovits' (so-vits-svc-fork)
TOTAL_EPOCHS = 300        # цель суммарно, а не за один прогон
BATCH_SIZE = 8            # 4, если не хватит памяти GPU
SAVE_EVERY = 25           # как часто писать чекпоинт
TIME_BUDGET_HOURS = 7.5   # запас до лимита сессии в 9 часов
TEXT = 'Проверка синтеза. Старинный замок на горе, а на двери замок.'


## 1. Окружение

В фоновом прогоне некому нажать «включить GPU», поэтому проверки падают сразу
и понятно.


In [ ]:
import os, pathlib, shutil, subprocess, sys, time, socket, textwrap

import torch
assert torch.cuda.is_available(), 'включите Accelerator -> GPU в панели справа'
try:
    socket.create_connection(('pypi.org', 443), timeout=10).close()
except OSError:
    raise SystemExit('включите Internet -> On в панели справа')
print('GPU:', torch.cuda.get_device_name(0))

WORK = pathlib.Path('/kaggle/working')
for name in ('profiles', 'voices', 'logs'):
    (WORK / name).mkdir(exist_ok=True)
STARTED = time.time()


## 2. Установка


In [ ]:
!git clone --depth 1 -b claude/voice-cloning-text-synthesis-7mnlwx https://github.com/tyetladd/RustTraining /kaggle/working/RustTraining 2>/dev/null || true
%pip install -q -e '/kaggle/working/RustTraining/voice-clone-tts[stress,asr,silero]'

if VC == 'rvc':
    !git clone --depth 1 https://github.com/IAHispano/Applio /kaggle/working/Applio 2>/dev/null || true
    %pip install -q -r /kaggle/working/Applio/requirements.txt
    # Сборки torch 2.11 с PyPI собраны под CUDA 13, а образ Kaggle не кладёт
    # libcudart.so.13 туда, где его ищет torchaudio: импорт падает, и обучение
    # не стартует. Переставляем пару под CUDA 12.8 из индекса PyTorch.
    %pip install -q -U --index-url https://download.pytorch.org/whl/cu128 torch==2.11.0 torchaudio==2.11.0
    os.environ['VCTTS_APPLIO_DIR'] = '/kaggle/working/Applio'

else:
    %pip install -q -U so-vits-svc-fork

!vctts converters


### Проверка окружения после установки

Ворох сообщений вида «X requires Y, but you have Z» выше — это pip, ругающийся на
*другие* пакеты образа (bigframes, ydata-profiling, google-colab и прочие), которыми
мы не пользуемся. Значение имеет одно: импортируются ли `torch` и `torchaudio` в
свежем процессе — именно так их увидит обучение, которое идёт отдельными процессами.

Ячейка ниже проверяет это и говорит, что делать, если нет.


In [ ]:
import json, subprocess, sys

PROBE = """
import json
out = {}
try:
    import torch
    out['torch'] = torch.__version__
    out['cuda'] = torch.cuda.is_available()
except Exception as exc:
    out['torch'] = f'ОШИБКА: {exc.__class__.__name__}: {exc}'
for name in ('torchaudio', 'torchvision'):
    try:
        out[name] = __import__(name).__version__
    except Exception as exc:
        out[name] = f'ОШИБКА: {exc.__class__.__name__}: {exc}'
print(json.dumps(out))
"""

# Важно: опрашиваем НОВЫЙ процесс. В текущем ядре torch уже импортирован, и оно
# покажет старую версию, даже если установка её заменила. Обучение тоже идёт в
# отдельных процессах — значит, видеть мы должны именно эту картину.
probe = subprocess.run([sys.executable, '-c', PROBE], capture_output=True, text=True)
fresh = json.loads(probe.stdout) if probe.stdout.strip().startswith('{') else {}
if not fresh:
    print('не удалось опросить окружение:\n', probe.stderr[-2000:])
else:
    for key, value in fresh.items():
        print(f'  {key:12}: {value}')

problems = [k for k, v in fresh.items() if str(v).startswith('ОШИБКА')]
if 'torchaudio' in problems:
    print('\n!! torchaudio не импортируется — Applio использует его для FCPE,')
    print('   обучение не стартует. Обычно это несовпадение сборки CUDA:')
    print('   %pip install -q -U --index-url https://download.pytorch.org/whl/cu128 \\')
    print('        torch==2.11.0 torchaudio==2.11.0')
elif 'torchvision' in problems:
    print('\n!! torchvision не совместим с установленным torch. Нашему пути он не нужен:')
    print('   %pip uninstall -y torchvision')
elif fresh.get('cuda') is False:
    print('\n!! CUDA не видна из свежего процесса — проверьте, что GPU включён')
else:
    print('\nокружение в порядке, можно продолжать')


## 3. Запись и состояние прошлого прогона

Запись берётся из подключённого датасета. Если вы подключили output прошлой
версии (**+ Add Data → Your Work**), его `logs/`, `profiles/` и `voices/`
копируются обратно в рабочую папку — и обучение продолжится, а не начнётся заново.


In [ ]:
INPUT = pathlib.Path('/kaggle/input')

audio = sorted(INPUT.rglob('*.mp3')) + sorted(INPUT.rglob('*.wav')) + sorted(INPUT.rglob('*.m4a'))
audio = [path for path in audio if 'dataset_raw' not in path.parts and path.stat().st_size > 10_000]
assert audio, 'подключите датасет с записью через + Add Data'
SOURCE = audio[0]
print('запись:', SOURCE, f'({SOURCE.stat().st_size / 1e6:.1f} МБ)')

# Состояние прошлой версии, если её подключили как входные данные.
for previous in sorted(INPUT.glob('*')):
    if not (previous / 'logs').exists():
        continue
    for name in ('logs', 'profiles', 'voices'):
        if (previous / name).exists():
            shutil.copytree(previous / name, WORK / name, dirs_exist_ok=True)
    print('подхвачено состояние из', previous)

checkpoints = list((WORK / 'logs' / SPEAKER).glob('*.pth')) if (WORK / 'logs' / SPEAKER).exists() else []
RESUME = bool(checkpoints)
print('режим:', 'продолжение' if RESUME else 'с нуля',
      f'({len(checkpoints)} чекпоинт(ов) найдено)')


### Что показывает запись

Вердикт по длительности, формату, уровню, клиппингу и SNR — до начала обучения.
Прогон не останавливается даже при `✗`: решать вам. Но если здесь написано
«лучше переписать», то часы GPU уйдут на запись, которую всё равно придётся
переписывать.


In [ ]:
cmd = f'vctts check "{SOURCE}" --purpose vc'
!{cmd}


## 4. Профиль диктора

Строится один раз; при продолжении переиспользуется.


In [ ]:
profile_dir = WORK / 'profiles' / SPEAKER
if (profile_dir / 'profile.json').exists():
    print('профиль уже есть, пропускаю')
else:
    cmd = f'vctts profile build "{SOURCE}" -o "{profile_dir}" --name {SPEAKER} --overwrite'
    !{cmd}


## 5. Обучение

Запускается через `subprocess`, чтобы ошибку или таймаут можно было поймать:
фоновый прогон, упавший с исключением, сохраняется как **failed**, и до output
можно не добраться. Поэтому обучение ограничено бюджетом времени, а любой
неуспех не роняет ноутбук — чекпоинты остаются в output, а следующая версия
продолжит с них.


In [ ]:
def train_command(resume: bool, timeout_seconds: int) -> list[str]:
    """Команда обучения; у драйверов разные полезные опции."""
    args = [
        'vctts', 'voice', 'train',
        '-p', str(WORK / 'profiles' / SPEAKER),
        '-o', str(WORK / 'voices' / SPEAKER),
        '--vc', VC,
        '--epochs', str(TOTAL_EPOCHS),
        '--resume' if resume else '--overwrite',
        '--vc-option', f'batch_size={BATCH_SIZE}',
        '--vc-option', f'timeout={timeout_seconds}',
    ]
    if VC == 'rvc':
        # Логи Applio должны лежать вне чекаута, чтобы попасть в output версии.
        args += ['--vc-option', f'logs_dir={WORK / "logs"}',
                 '--vc-option', f'save_every_epoch={SAVE_EVERY}']
    else:
        args += ['--sample-rate', '44100']
    return args


budget = int(TIME_BUDGET_HOURS * 3600 - (time.time() - STARTED))
command = train_command(RESUME, budget)
print(' '.join(command), '\n')

result = subprocess.run(command, text=True)
TRAINED = result.returncode == 0
print('\nобучение:', 'завершено' if TRAINED else
      f'прервано (код {result.returncode}) — чекпоинты сохранены, продолжите новой версией')


## 6. Проверка: Silero TTS + ваш голос

Работает только если модель уже собрана. Если обучение прервалось по времени,
ячейка просто сообщит об этом.


In [ ]:
model_dir = WORK / 'voices' / SPEAKER
sample = WORK / f'{SPEAKER}-sample.wav'

if (model_dir / 'voice_model.json').exists():
    speak = [
        'vctts', 'speak', '-b', 'silero', '-l', 'ru',
        '--voice-model', str(model_dir), '--transpose', 'auto',
        '-t', TEXT, '-o', str(sample),
    ]
    check = subprocess.run(speak, text=True)
    print('синтез:', 'ок' if check.returncode == 0 else 'не удался')
else:
    print('модели ещё нет — обучение не дошло до конца, продолжите новой версией')

if sample.exists():
    from IPython.display import Audio, display
    display(Audio(str(sample)))


## 7. Итог и что забрать

Всё, что осталось в `/kaggle/working`, попадает в output сохранённой версии:
оттуда можно скачать архив модели, а можно подключить этот output к следующему
запуску — тогда ноутбук сам продолжит обучение.


In [ ]:
# Чекаут Applio и клон репозитория в output не нужны — они занимают гигабайты.
for junk in ('Applio', 'RustTraining'):
    shutil.rmtree(WORK / junk, ignore_errors=True)

if (model_dir / 'voice_model.json').exists():
    archive = shutil.make_archive(str(WORK / f'{SPEAKER}-voice'), 'zip', model_dir)
    print('архив модели:', archive)

elapsed = (time.time() - STARTED) / 3600
print(textwrap.dedent(f'''
    итог прогона
      голос      : {SPEAKER} ({VC})
      режим      : {'продолжение' if RESUME else 'с нуля'}
      обучение   : {'завершено' if TRAINED else 'прервано, нужен ещё прогон'}
      время      : {elapsed:.1f} ч
      дальше     : {'скачайте архив из output' if TRAINED else 'запустите Save & Run All ещё раз, подключив этот output через + Add Data -> Your Work'}
'''))


---

### Если хочется следить за ходом

Открытая вкладка для фонового прогона не нужна: состояние видно в **Notebook →
версии** (Running / Complete / Failed), а полный лог — по клику на версию.
Kaggle присылает уведомление о завершении. Квота расходуется только фактическим
временем прогона.

### Если прогон упал на установке

Самое частое — Applio ставит свои версии torch и transformers поверх образа.
В фоновом прогоне это обычно проходит; если нет, попробуйте `VC = 'sovits'`:
so-vits-svc-fork ставится одной командой и не конфликтует с образом.
